# 3. Database Security — SQL, Cosmos DB, PostgreSQL

Databases are where the crown-jewel data lives. The AZ-500 exam treats Azure
SQL as the "main event", but also expects you to know the equivalent controls
in **Cosmos DB**, **Azure Database for PostgreSQL / MySQL**, and how to defend
data with features like **TDE**, **Always Encrypted**, **dynamic masking**, and
**row-level security**.

### What you'll learn
1. The **defence-in-depth model** for Azure SQL.
2. **Entra ID authentication** for SQL and the PaaS DBs.
3. **Firewall rules & private endpoints** — the network layer.
4. **Dynamic Data Masking** — hide PII at query time.
5. **Row-Level Security (RLS)** — let users see only their own rows.
6. **TDE vs Always Encrypted** — two very different things.
7. **Auditing & Defender for SQL** — detect misuse.
8. **Cosmos DB / PostgreSQL security** — the same ideas in a different dialect.
9. **Bad → best progression** on a real SQL deployment.

### Analogy 🏦
A database is like a bank vault:
- **Firewall + private endpoint** = who is allowed in the bank lobby.
- **Entra auth** = showing ID to the teller (vs a shared vault password).
- **Row-level security** = you can only open your own safety-deposit box.
- **Dynamic masking** = the teller shows you only the last 4 digits of an account.
- **TDE** = the vault walls are made of reinforced concrete.
- **Always Encrypted** = the documents inside are in a *second* locked box that
  only the customer has the key for — even the bank can't read them.

## Before you run this notebook

1. Run `uv sync` in the lab folder.
2. Pick the `.venv` kernel in VS Code.
3. Reload the window if it doesn't appear.

All demos are plain-Python simulations — no SQL Server needed.

## 1. Azure SQL defence-in-depth — five layers

```
┌── 1. Network ───────────────────────────────────────────────┐
│   Firewall rules · Private Endpoint · VNet service rules    │
│ ┌── 2. Authentication ─────────────────────────────────────┐│
│ │   Entra ID (preferred) · SQL auth (legacy)                ││
│ │ ┌── 3. Authorization ────────────────────────────────────┐││
│ │ │   DB roles · Row-Level Security · least-privilege      │││
│ │ │ ┌── 4. Data protection ────────────────────────────────┐││
│ │ │ │   TDE · Always Encrypted · Dynamic Data Masking      │││
│ │ │ └─────────────────────────────────────────────────────┘││
│ │ └───────────────────────────────────────────────────────┘││
│ └─────────────────────────────────────────────────────────┘│
│   5. Monitoring  →  Auditing · Defender for SQL · Sentinel  │
└─────────────────────────────────────────────────────────────┘
```

Every exam question maps to one of these five layers. When you read a scenario,
ask: *"which layer is being asked about?"*

## 2. Entra ID authentication for SQL

```bash
# Set an Entra GROUP as SQL admin (NOT an individual — that person leaves one day)
az sql server ad-admin create -g rg-prod -s sql-prod \
  --display-name 'DBA Team' --object-id <group-object-id>

# (Optional) Disable SQL authentication entirely
az sql server update -g rg-prod -n sql-prod \
  --enable-ad-only-auth true
```

From the app side, nothing but a connection-string change:

```text
Server=tcp:sql-prod.database.windows.net,1433;
Database=mydb;
Authentication=Active Directory Default;   <-- uses Managed Identity / Azure CLI / VS Code
Encrypt=True;
```

> **Exam tip**: *Entra-only authentication* is a toggle. Turning it on is the
> single biggest improvement most SQL servers can make.

## 3. Firewall rules & Private Endpoints — the network layer

| Control | What it does |
|---------|-------------|
| **Server-level firewall**        | IP allowlist applied to the logical server |
| **Database-level firewall**      | IP allowlist per database (overrides server) |
| **Allow Azure services** toggle  | ⚠️ **Off** by default now — on = "any Azure subscription in the world" |
| **VNet service endpoint**        | Allow a subnet by identity, not IP |
| **Private Endpoint**             | The server gets a private IP inside your VNet |
| **Public network access**        | When Disabled, only Private Endpoint works |

```bash
# Best practice: disable public network, add a private endpoint
az sql server update -g rg-prod -n sql-prod \
  --set publicNetworkAccess=Disabled

az network private-endpoint create -g rg-prod -n pe-sql \
  --vnet-name vnet-prod --subnet subnet-pe \
  --private-connection-resource-id /subscriptions/.../servers/sql-prod \
  --group-id sqlServer --connection-name pe-sql-conn
```

In [ ]:
# Simulate the full connection-time check
SQL_SERVER = {
    'public_network': False,       # best practice: off
    'entra_only': True,
    'firewall_rules': [('office',  '203.0.113.0', '203.0.113.255')],
    'vnet_rules':    ['subnet-app'],
    'private_endpoint': True,
    'allow_azure_services': False,
}

def ip_between(ip, lo, hi):
    import ipaddress
    return ipaddress.ip_address(lo) <= ipaddress.ip_address(ip) <= ipaddress.ip_address(hi)

def connect(req):
    # network
    if req.get('via_private_endpoint') and SQL_SERVER['private_endpoint']:
        pass                        # fine
    elif not SQL_SERVER['public_network']:
        return '❌ public network disabled'
    else:
        if req.get('subnet') in SQL_SERVER['vnet_rules']:
            pass
        elif req.get('src_ip') and any(ip_between(req['src_ip'], a, b)
                                       for _, a, b in SQL_SERVER['firewall_rules']):
            pass
        elif req.get('from_azure') and SQL_SERVER['allow_azure_services']:
            pass
        else:
            return '❌ blocked by SQL firewall'
    # auth
    if SQL_SERVER['entra_only'] and req['auth'] == 'sql':
        return '❌ Entra-only mode — SQL auth rejected'
    return f'✅ connected as {req["principal"]} ({req["auth"]})'

requests = [
    {'label': 'app (MI) via private endpoint',
     'via_private_endpoint': True, 'auth': 'entra', 'principal': 'mi-payroll'},
    {'label': 'DBA from office IP',
     'src_ip': '203.0.113.20', 'auth': 'entra', 'principal': 'alice@contoso.com'},
    {'label': 'coffee-shop laptop, SQL auth',
     'src_ip': '185.1.2.3', 'auth': 'sql', 'principal': 'sa'},
    {'label': 'legacy tool using SQL auth via PE',
     'via_private_endpoint': True, 'auth': 'sql', 'principal': 'etl_user'},
]
for r in requests:
    print(f'{r["label"]:45s} → {connect(r)}')

## 4. Dynamic Data Masking (DDM)

Masking **replaces sensitive values at query time** so support staff, analysts,
and developers see realistic-but-fake data instead of real PII. The underlying
data is **not** encrypted — a determined admin can still bypass it.

| Mask function | Example input | Example output |
|---------------|--------------|----------------|
| `default`     | anything      | `XXXX` / `0` / `1900-01-01` |
| `email`       | `alice@contoso.com` | `aXXX@XXXX.com` |
| `partial(0,4,"XXX-XXX-")` | `555-123-5678` | `XXX-XXX-5678` |
| `random(min,max)`         | `145000` | `147891` (random) |

```sql
-- T-SQL
ALTER TABLE dbo.Customers
  ALTER COLUMN Email ADD MASKED WITH (FUNCTION='email()');

ALTER TABLE dbo.Customers
  ALTER COLUMN SSN ADD MASKED WITH (FUNCTION='default()');

-- Grant an analyst normal SELECT (they'll see masked data)
GRANT SELECT ON dbo.Customers TO analyst_user;
-- Give DPO the UNMASK privilege (they see plaintext)
GRANT UNMASK TO dpo_user;
```

In [ ]:
# Dynamic masking simulator — shows how three different users see the same row
import random

MASK_RULES = {
    'email':       {'fn': 'email'},
    'credit_card': {'fn': 'partial', 'prefix': 0, 'suffix': 4, 'padding': 'XXXX-XXXX-XXXX-'},
    'ssn':         {'fn': 'default'},
    'phone':       {'fn': 'partial', 'prefix': 0, 'suffix': 4, 'padding': 'XXX-XXX-'},
    'salary':      {'fn': 'random', 'min': 30000, 'max': 200000},
}
ROW = {'name': 'Alice Johnson', 'email': 'alice.johnson@contoso.com',
       'credit_card': '4111-1111-1111-1234', 'ssn': '123-45-6789',
       'phone': '555-123-5678', 'salary': 145000}

def apply_mask(val, rule):
    fn = rule['fn']
    if fn == 'default':
        return 'XXXX' if isinstance(val, str) else 0
    if fn == 'email':
        local, domain = str(val).split('@', 1)
        return f'{local[0]}XXX@XXXX.com'
    if fn == 'partial':
        s = str(val)
        return rule['padding'] + s[-rule['suffix']:]
    if fn == 'random':
        random.seed(hash(val))          # deterministic for demo output
        return random.randint(rule['min'], rule['max'])
    return val

def query_as(role, row):
    if role in ('db_owner', 'dpo_user'):      # have UNMASK
        return row
    return {k: apply_mask(v, MASK_RULES[k]) if k in MASK_RULES else v
            for k, v in row.items()}

for role in ['db_owner', 'analyst', 'customer_service']:
    print(f'--- As {role} ---')
    print(' ', query_as(role, ROW))
print('\n⚠️ Masking is NOT a security boundary: repeated WHERE clauses can reveal values.')

## 5. Row-Level Security (RLS) — each user sees only their rows

RLS lets you define a **filter predicate** — a function that returns 1 for
rows the current session user may see, 0 otherwise. It runs on every query.

```sql
-- 1. Filter function: tenant_id must match the SESSION_CONTEXT / USER_NAME
CREATE FUNCTION dbo.fn_tenantFilter(@tenant nvarchar(50))
    RETURNS TABLE WITH SCHEMABINDING
AS RETURN SELECT 1 AS ok
   WHERE @tenant = CAST(SESSION_CONTEXT(N'tenant_id') AS nvarchar(50));

-- 2. Security policy binds the function to the table
CREATE SECURITY POLICY dbo.TenantPolicy
  ADD FILTER PREDICATE dbo.fn_tenantFilter(tenant_id) ON dbo.Orders,
  ADD BLOCK PREDICATE  dbo.fn_tenantFilter(tenant_id) ON dbo.Orders AFTER INSERT
WITH (STATE = ON);
```

The app sets the tenant on each connection:

```sql
EXEC sp_set_session_context 'tenant_id', N'acme-corp';
SELECT * FROM Orders;     -- only acme-corp rows come back
```

In [ ]:
# Simulate RLS for a multi-tenant Orders table
ORDERS = [
    {'id': 1, 'tenant_id': 'acme',   'amount': 120},
    {'id': 2, 'tenant_id': 'acme',   'amount': 450},
    {'id': 3, 'tenant_id': 'globex', 'amount':  80},
    {'id': 4, 'tenant_id': 'initech','amount':  30},
]

def select_orders(session_tenant, is_dbo=False):
    if is_dbo:
        return ORDERS                              # DBO bypasses RLS by default
    return [r for r in ORDERS if r['tenant_id'] == session_tenant]

print('Acme app sees:   ', select_orders('acme'))
print('Globex app sees: ', select_orders('globex'))
print('Initech app sees:', select_orders('initech'))
print('DBO (no filter): ', select_orders(None, is_dbo=True))
print('\n💡 Never let the app connect as DBO — that bypasses RLS!')

## 6. TDE vs Always Encrypted — the classic exam question

| | **TDE**  (Transparent Data Encryption) | **Always Encrypted** |
|-|------------------------------------------|----------------------|
| Protects against | Stolen disk / backup                   | DBAs, cloud operators, compromised SQL engine, SQL injection of SELECT |
| Encryption scope | Entire database files                  | Specific columns |
| Where decryption happens | Inside the SQL engine          | In the **client driver** (ADO.NET, JDBC, ODBC) |
| Who holds the key | SQL Server / Key Vault (BYOK)         | Client application only |
| Query capability | Full SQL                               | Equality only (deterministic) or none (randomized) |
| Performance cost | Tiny                                    | Larger — encrypted indexes, extra round-trips |

**Mental rule:** *If the DBA shouldn't see the plaintext → **Always Encrypted**.
If you only care about stolen backups → **TDE***.

```bash
# TDE is ON by default for Azure SQL. Switch to your own key (BYOK):
az sql server key create -g rg-prod -s sql-prod \
  --kid https://my-kv.vault.azure.net/keys/sql-tde/<version>

az sql server tde-key set -g rg-prod -s sql-prod \
  --server-key-type AzureKeyVault \
  --kid https://my-kv.vault.azure.net/keys/sql-tde/<version>
```

```sql
-- Always Encrypted: declare keys, then mark columns
CREATE COLUMN MASTER KEY CMK1 WITH (
  KEY_STORE_PROVIDER_NAME = N'AZURE_KEY_VAULT',
  KEY_PATH = N'https://my-kv.vault.azure.net/keys/ae-cmk/<ver>');

CREATE COLUMN ENCRYPTION KEY CEK1
  WITH VALUES (COLUMN_MASTER_KEY = CMK1,
               ALGORITHM = 'RSA_OAEP',
               ENCRYPTED_VALUE = 0x01C5...);

ALTER TABLE Patients
  ALTER COLUMN SSN char(11) COLLATE Latin1_General_BIN2
  ENCRYPTED WITH (COLUMN_ENCRYPTION_KEY = CEK1,
                  ENCRYPTION_TYPE = DETERMINISTIC,
                  ALGORITHM = 'AEAD_AES_256_CBC_HMAC_SHA_256') NOT NULL;
```

In [ ]:
# Show the difference: what does each component see?
import base64, os, hashlib

class ColumnEncryptionKey:
    def __init__(self):
        self.cek = os.urandom(32)           # stays in the CLIENT only
    def enc(self, plaintext: str) -> bytes:
        # DETERMINISTIC (toy): same plaintext → same ciphertext (so equality works)
        digest = hashlib.sha256(self.cek + plaintext.encode()).digest()
        return digest                        # pretend this is the ciphertext
    def dec(self, ct: bytes) -> str:
        return '<decrypted client-side>'

client_cek = ColumnEncryptionKey()

# The CLIENT encrypts before sending INSERT
row_on_wire = {
    'patient_id': 42,
    'name': 'Jane Doe',                                            # plain
    'ssn_enc':  base64.b64encode(client_cek.enc('123-45-6789')).decode(),
}
print('What the SQL engine / DBA / TDE backup sees:')
print(' ', row_on_wire)

print('\nWhat the authorised CLIENT (with the CEK) sees after decrypt:')
print('  {"patient_id":42, "name":"Jane Doe", "ssn": "123-45-6789"}')

print('\nEquality query still works because DETERMINISTIC encryption is deterministic:')
print('  WHERE ssn_enc =', base64.b64encode(client_cek.enc('123-45-6789')).decode()[:16], '...')

## 7. Auditing & Defender for SQL

```bash
# Stream audit logs to Log Analytics (best for KQL + Sentinel)
az sql server audit-policy update -g rg-prod -n sql-prod \
  --state Enabled \
  --log-analytics-target-state Enabled \
  --log-analytics-workspace-resource-id /subscriptions/.../workspaces/la-sec

# Turn on Defender for SQL (vulnerability assessment + advanced threat detection)
az sql server ms-support update -g rg-prod -n sql-prod --status Enabled
az security atp-cloud sql-server update -g rg-prod -s sql-prod --is-enabled true
```

Audit destinations — pick per use-case:

| Destination       | Good for |
|-------------------|----------|
| **Storage account**  | Long-term, cheapest, raw XEL files |
| **Log Analytics**    | KQL queries, Sentinel integration |
| **Event Hub**        | Stream to external SIEM / Splunk |

**Defender for SQL** gives you:
- **Vulnerability assessment**: weekly scan, suggests fixes (missing TDE, overly broad firewall).
- **Advanced Threat Protection**: detects SQL injection, anomalous logins, exfil patterns.

## 8. Other PaaS databases — same ideas, different knobs

### Cosmos DB
- **Entra RBAC for data plane**: `az cosmosdb sql role assignment create` — prefer over the *Primary Key*.
- **Primary keys**: like storage account keys → rotate them; use read-only keys when possible.
- **Network**: IP firewall + VNet rule + Private Endpoint (same pattern).
- **CMK encryption**: must be enabled at account creation; uses Key Vault.
- **Always Encrypted** is also available for Cosmos DB (client-side).

### Azure Database for PostgreSQL / MySQL (Flexible Server)
- **Entra authentication**: enable it *in addition to* or *instead of* the built-in admin.
- **SSL enforcement**: `require_secure_transport=ON` (default).
- **Private access**: deploy the server into a delegated subnet (VNet-integrated) or use Private Link.
- **CMK**: BYOK via Key Vault.
- **Microsoft Defender for open-source databases** catches anomalies like MySQL injection.

> 🎯 On the exam, the phrase *"without managing keys in the application"* almost
> always points to **managed identity + Entra authentication**, regardless of
> which database engine is in the question.

## 9. Bad → best — a full SQL deployment

```bash
# ❌ BAD: public network, SQL auth only, no auditing
az sql server create -g rg-prod -n sql-bad -l eastus \
  -u sqladmin -p 'S3cret!Password'                   # password in shell history!

# 🟡 BETTER: Entra-only, TDE with CMK, auditing to LA, firewall IP allowlist
az sql server update -g rg-prod -n sql-better --enable-ad-only-auth true
az sql server audit-policy update ... --log-analytics-target-state Enabled

# ✅ BEST: private endpoint, no public network, Defender for SQL, CMK,
#          dynamic masking on PII, RLS for multi-tenant, Always Encrypted for SSNs
az sql server update -g rg-prod -n sql-best \
  --set publicNetworkAccess=Disabled \
  --enable-ad-only-auth true
az security atp-cloud sql-server update -g rg-prod -s sql-best --is-enabled true
# (private endpoint, TDE-with-CMK, masking, RLS, AE shown in earlier sections)
```

---
## Summary

| Layer | Control | Best practice |
|-------|---------|--------------|
| Network         | Firewall / Private Endpoint       | Public network Disabled, PE only |
| Authentication  | Entra ID / Managed Identity       | Entra-only mode, no SQL auth |
| Authorization   | DB roles, RLS                     | Least-priv; app user ≠ dbo |
| Data at rest    | TDE with BYOK                     | CMK in Key Vault; rotate annually |
| Data in use     | Always Encrypted, Dynamic Masking | AE for columns DBAs must not see |
| Monitoring      | Auditing, Defender for SQL        | Stream to Log Analytics + Sentinel |

**Next lab**: [04 — Defender for Cloud & Sentinel](../../04-defender-and-sentinel/)